# Session 6 — GPU: Model Depth and Memory Limits

## Background

Sessions 1–5 measured a single Transformer encoder block. Production models stack many: BERT-base has 12 layers, GPT-2 up to 48. Activation memory — tensors saved during the forward pass for backpropagation — scales linearly with depth. At some stack depth and batch size, the GPU's 23.7 GB GDDR6 fills up. The question is not whether OOM eventually happens, but where the boundary is.

## Goal

Find the GPU memory limit by sweeping `n_layers` at `batch=64`. Catch `CUDA out of memory` errors and record the exact OOM boundary. Track VRAM with `torch.cuda.memory_allocated()`. Produce `results/gpu_depth.json` for comparison in `03-analysis.ipynb`.

## Question

How many Transformer layers can the NVIDIA L4 sustain at `batch=64` before running out of memory?

---

**Runtime:** GPU (NVIDIA L4 or equivalent)

After running, results are saved to `results/gpu_depth.json` and loaded automatically by `03-analysis.ipynb`.

In [1]:
import time, torch, torch.nn as nn

assert torch.cuda.is_available(), "No GPU found. Runtime → Change runtime type → GPU."

device = torch.device("cuda")
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Device  : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {total_vram_gb:.1f} GB")
print(f"PyTorch : {torch.__version__}")

Device  : NVIDIA L4
VRAM    : 23.7 GB
PyTorch : 2.10.0+cu128


In [2]:
import sys; sys.path.insert(0, "..")
from transformer_block import BenchmarkTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD

# ── Session 6 config ──────────────────────────────────────────────────────────
BATCH_SIZE  = 64
SEQ_LEN     = 128
N_LAYERS_SWEEP = [1, 2, 4, 6, 8, 12, 16, 24]
WARMUP      = 5

# Fewer steps at deeper models — the backward pass grows linearly with depth.
def steps_for(n_layers):
    if n_layers <= 4:  return 30
    if n_layers <= 12: return 20
    return 15

class DeepTransformerModel(nn.Module):
    """Stack of N BenchmarkTransformerBlock instances."""
    def __init__(self, n_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([
            BenchmarkTransformerBlock(
                d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD
            )
            for _ in range(n_layers)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

print(f"Config  : batch={BATCH_SIZE}, seq_len={SEQ_LEN}, d_model={D_MODEL}")
print(f"Sweep   : n_layers = {N_LAYERS_SWEEP}")
print("DeepTransformerModel defined.")

Config  : batch=64, seq_len=128, d_model=512
Sweep   : n_layers = [1, 2, 4, 6, 8, 12, 16, 24]
DeepTransformerModel defined.


In [3]:
def benchmark(n_layers):
    """Returns (throughput_or_OOM_str, peak_vram_mb_or_None)."""
    n_steps = steps_for(n_layers)
    try:
        model     = DeepTransformerModel(n_layers).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        criterion = nn.MSELoss()
        model.train()
        torch.cuda.reset_peak_memory_stats(device)

        elapsed = 0.0
        for step in range(n_steps + WARMUP):
            x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
            y = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)

            torch.cuda.synchronize()
            t0 = time.perf_counter()
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            torch.cuda.synchronize()
            t1 = time.perf_counter()

            if step >= WARMUP:
                elapsed += t1 - t0

        throughput   = (n_steps * BATCH_SIZE) / elapsed
        peak_vram_mb = torch.cuda.max_memory_allocated(device) / 1e6
        torch.cuda.empty_cache()
        del model
        return round(throughput, 1), round(peak_vram_mb, 1)

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return "OOM", None

print("Benchmark function defined.")

Benchmark function defined.


In [4]:
results = {}

print(f"{'n_layers':<10}  {'steps':>6}  {'throughput':>15}  {'peak VRAM':>12}")
print(f"{'─'*10}  {'─'*6}  {'─'*15}  {'─'*12}")

oom_hit = False
for nl in N_LAYERS_SWEEP:
    if oom_hit:
        # Skip remaining sizes once OOM is confirmed
        results[str(nl)] = "OOM"
        print(f"{nl:<10}  {'—':>6}  {'OOM (skipped)':>15}  {'—':>12}")
        continue

    tput, vram = benchmark(nl)
    if tput == "OOM":
        results[str(nl)] = "OOM"
        oom_hit = True
        print(f"{nl:<10}  {steps_for(nl):>6}  {'OOM':>15}  {'—':>12}")
    else:
        results[str(nl)] = {"throughput": tput, "peak_vram_mb": vram}
        pct = vram / (total_vram_gb * 1000) * 100
        print(f"{nl:<10}  {steps_for(nl):>6}  {tput:>14,.0f}  {vram:>8,.0f} MB  ({pct:.0f}%)")

print()
oom_layers = [nl for nl in N_LAYERS_SWEEP if results.get(str(nl)) == "OOM"]
if oom_layers:
    print(f"OOM boundary: first OOM at n_layers={oom_layers[0]}")
else:
    print("No OOM observed across the full sweep — L4 headroom exceeded expectations.")

n_layers     steps       throughput     peak VRAM
──────────  ──────  ───────────────  ────────────


1               30           2,635       531 MB  (2%)


2               30           1,244       890 MB  (4%)


4               30             599     1,606 MB  (7%)


6               20             393     2,323 MB  (10%)


8               20             294     3,039 MB  (13%)


12              20             199     4,473 MB  (19%)


16              15             152     5,891 MB  (25%)


24              15             101     8,772 MB  (37%)

No OOM observed across the full sweep — L4 headroom exceeded expectations.


In [5]:
import json, os
from datetime import datetime, timezone

os.makedirs("results", exist_ok=True)
payload = {
    "hardware":     "GPU",
    "device_name":  torch.cuda.get_device_name(0),
    "session":      "6",
    "batch_size":   BATCH_SIZE,
    "seq_len":      SEQ_LEN,
    "n_layers_sweep": N_LAYERS_SWEEP,
    "timestamp":    datetime.now(timezone.utc).isoformat(),
    "results":      results,
}
path = "results/gpu_depth.json"
with open(path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved → {path}")

Saved → results/gpu_depth.json
